# 附录 E：LoRA 参数高效微调（Low-Rank Adaptation）

> ch06/ch07 的微调都要更新大量参数。LoRA 把可训练参数压到 **< 1%**，却能达到接近全量微调的效果——这是当前工业界最主流的微调技术。

## 核心思想

全量微调要更新权重矩阵 W（d×k）。LoRA 的洞察：**微调时的权重变化 ΔW 是低秩的**，可以用两个小矩阵的乘积近似：

$$\Delta W = B \cdot A \quad (B: d{\times}r,\ A: r{\times}k,\ r \ll d,k)$$

冻住原始 W，只训练 A 和 B。

| | 全量微调 | LoRA (r=8) |
|---|---|---|
| 可训练参数 | W 的 d×k 个 | A+B 的 (d+k)×r 个 |
| 768×768 层 | 590K | 12K（**2%**）|

> 为什么有效？因为「学会一个新任务」所需的权重变化本就在低维子空间里，不需要动用全部参数。

## 1. LoRA 层实现

关键：B 初始化为零（训练初期 ΔW=0，不破坏原模型），A 正常初始化。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LoRALayer(nn.Module):
    """给一个 nn.Linear 加 LoRA 旁路：冻结原权重，训练低秩 A、B。"""

    def __init__(self, layer, r=8, alpha=16):
        super().__init__()
        self.layer = layer
        # 冻结原始权重
        for p in layer.parameters():
            p.requires_grad = False
        in_f = layer.in_features
        out_f = layer.out_features
        self.scaling = alpha / r
        # A 正常初始化，B 零初始化（关键：保证初期 ΔW=0）
        self.A = nn.Parameter(torch.randn(in_f, r) * 0.01)
        self.B = nn.Parameter(torch.zeros(r, out_f))

    def forward(self, x):
        # 原始输出 + LoRA 旁路：xW + scaling * x(BA)
        return self.layer(x) + self.scaling * (x @ (self.A @ self.B))


# 验证：初始时 LoRA 不改变输出
orig = nn.Linear(128, 128)
lora = LoRALayer(orig, r=8, alpha=16)
x = torch.randn(2, 16, 128)
diff = (lora(x) - orig(x)).abs().max().item()
print(f"初始 LoRA 与原模型输出差异: {diff}（应=0，因 B 初始化为零）")
orig_params = 128 * 128
lora_params = (128 + 128) * 8
print(f"原参数 {orig_params:,} → LoRA 新增 {lora_params:,}（{100*lora_params/orig_params:.1f}%）")

## 2. 把 GPT 的所有 Linear 替换为 LoRA

In [ ]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, r=8, alpha=16):
        super().__init__()
        self.lora = LoRALayer(linear, r=r, alpha=alpha)
    def forward(self, x):
        return self.lora(x)

def replace_linear_with_lora(module, r=8, alpha=16):
    """递归把 module 下所有 nn.Linear 替换为 LoRA 版。"""
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LinearWithLoRA(child, r, alpha))
        else:
            replace_linear_with_lora(child, r, alpha)


from src.gpt import GPTModel, GPT_CONFIG_124M

cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 32})
torch.manual_seed(123)
model = GPTModel(cfg)
model.out_head = nn.Linear(cfg["emb_dim"], 2)  # 分类头

# 关键：先冻结所有参数，再给 trf_blocks 加 LoRA
for p in model.parameters():
    p.requires_grad = False
replace_linear_with_lora(model.trf_blocks, r=8, alpha=16)
# 分类头保持可训练
for p in model.out_head.parameters():
    p.requires_grad = True

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数: {total:,}")
print(f"可训练: {trainable:,} ({100*trainable/total:.2f}%)  ← LoRA 把训练参数压到 < 1%")
print(f"冻结:   {total-trainable:,} ({100*(total-trainable)/total:.2f}%)")

## 3. 用 LoRA 做分类微调（复用 ch06 任务）

In [ ]:
import tiktoken

# 自造情感数据（同 ch06）
POS = ["这部电影非常精彩 我很喜欢", "太好看了 剧情感人至深", "画面优美 值得推荐",
       "演技出色 故事动人", "完美之作 强烈推荐", "音乐动听 视觉震撼",
       "节奏紧凑 引人入胜", "结局温暖 回味无穷"]
NEG = ["太糟糕了 浪费时间", "剧情无聊 让人失望", "画面粗糙 毫无诚意",
       "演技尴尬 故事混乱", "简直烂片 不忍直视", "噪音刺耳 看不下去",
       "节奏拖沓 昏昏欲睡", "结局糟糕 一无是处"]
tok = tiktoken.get_encoding("gpt2")
data = []
for t in POS: data.append((tok.encode(t)[:32], 1))
for t in NEG: data.append((tok.encode(t)[:32], 0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# 只优化可训练参数（LoRA + 分类头）
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.1
)

model.train()
for epoch in range(15):
    total = 0
    for ids, label in data:
        x = torch.tensor([ids + [50256]*(32-len(ids))]).to(device)
        y = torch.tensor([label]).to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x)[:, -1, :], y)
        loss.backward(); optimizer.step()
        total += loss.item()
    if epoch % 3 == 0 or epoch == 14:
        print(f"epoch {epoch}: loss {total/len(data):.4f}")

# 评估
model.eval()
correct = 0
with torch.no_grad():
    for ids, label in data:
        x = torch.tensor([ids + [50256]*(32-len(ids))]).to(device)
        pred = model(x)[:, -1, :].argmax(-1)
        correct += (pred.item() == label)
print(f"\nLoRA 微调后准确率: {100*correct/len(data):.0f}%（只动了 {100*trainable/total:.2f}% 参数）")

---
> **小结**：LoRA 用低秩矩阵 BA 旁路冻结的权重，可训练参数压到 < 1%，效果接近全量微调。
> **优势**：省显存（可在单卡微调 70B）、可叠加（为不同任务训练不同 LoRA，切换即用）、不破坏原模型。
> **α/r 缩放**：alpha 控制 LoRA 更新的强度，通常 alpha=2r。